# IV with Weak Instruments: 2SLS vs LIML vs Fuller

This tutorial demonstrates instrumental variables estimation with a focus on:
1. **Weak instrument diagnostics** (F-statistic, Stock-Yogo)
2. **Comparing estimators**: 2SLS, LIML, Fuller
3. **Anderson-Rubin confidence intervals** (weak-IV robust)
4. **When 2SLS fails** and alternatives succeed

**Scenario**: Returns to education using draft lottery as instrument.
- Endogenous variable: Years of schooling
- Instrument: Draft lottery number
- Outcome: Log wages

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
sys.path.insert(0, '../../src')

from causal_inference.iv import iv_2sls, liml, fuller, anderson_rubin_ci
from causal_inference.iv import weak_iv_test, stock_yogo_test

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## Data Generating Process

We create a function to generate IV data with controllable first-stage strength.

In [ ]:
def generate_iv_data(n=1000, first_stage_strength=0.3, true_effect=0.1, seed=42):
    """
    Generate IV data with specified first-stage strength.
    
    Parameters
    ----------
    n : int
        Sample size
    first_stage_strength : float
        Coefficient of Z on D (higher = stronger instrument)
    true_effect : float
        True causal effect of D on Y
    
    Returns
    -------
    Y, D, Z, X : arrays
        Outcome, treatment, instrument, covariates
    """
    np.random.seed(seed)
    
    # Instrument (draft lottery number, standardized)
    Z = np.random.randn(n)
    
    # Covariates
    X = np.random.randn(n, 2)
    
    # Unobserved confounder (ability)
    U = np.random.randn(n)
    
    # Endogenous treatment (education)
    D = first_stage_strength * Z + 0.3 * X[:, 0] + 0.5 * U + np.random.randn(n) * 0.5
    
    # Outcome (wages)
    Y = true_effect * D + 0.2 * X[:, 0] + 0.3 * X[:, 1] + 0.4 * U + np.random.randn(n) * 0.5
    
    return Y, D, Z, X

# Generate data with STRONG instrument
Y_strong, D_strong, Z_strong, X_strong = generate_iv_data(
    n=1000, first_stage_strength=0.5, true_effect=0.1
)

# Generate data with WEAK instrument
Y_weak, D_weak, Z_weak, X_weak = generate_iv_data(
    n=1000, first_stage_strength=0.1, true_effect=0.1
)

print("Data generated with:")
print(f"  True effect: 0.1")
print(f"  Strong instrument: first_stage = 0.5")
print(f"  Weak instrument: first_stage = 0.1")

---

## Part 1: Weak Instrument Diagnostics

The **first-stage F-statistic** measures instrument strength:
- F > 10: Strong instrument (rule of thumb)
- F < 10: Weak instrument concerns

The **Stock-Yogo test** provides formal critical values.

In [ ]:
# Test strong instrument
print("=== Strong Instrument ===")
result_strong = iv_2sls(Y_strong, D_strong, Z_strong, X_strong)
print(f"First-stage F-statistic: {result_strong['first_stage']['f_statistic']:.2f}")
print(f"Status: {'STRONG' if result_strong['first_stage']['f_statistic'] > 10 else 'WEAK'}")

print("\n=== Weak Instrument ===")
result_weak = iv_2sls(Y_weak, D_weak, Z_weak, X_weak)
print(f"First-stage F-statistic: {result_weak['first_stage']['f_statistic']:.2f}")
print(f"Status: {'STRONG' if result_weak['first_stage']['f_statistic'] > 10 else 'WEAK'}")

In [ ]:
# Stock-Yogo test
print("\n=== Stock-Yogo Critical Values ===")
sy_result = stock_yogo_test(result_weak['first_stage']['f_statistic'], n_instruments=1)

print(f"\nF-statistic: {result_weak['first_stage']['f_statistic']:.2f}")
print(f"\nMaximum relative bias thresholds:")
for bias, cv in sy_result['critical_values'].items():
    status = '✓' if result_weak['first_stage']['f_statistic'] > cv else '✗'
    print(f"  {bias}: {cv:.2f} {status}")

---

## Part 2: Comparing Estimators

We compare three IV estimators:

1. **2SLS**: Standard estimator, but biased toward OLS with weak instruments
2. **LIML**: Limited Information Maximum Likelihood, less biased with weak IV
3. **Fuller**: Modified LIML with finite-sample adjustment (α=1 or α=4)

In [ ]:
true_effect = 0.1

print("=== Comparison with STRONG Instrument ===")
print(f"True effect: {true_effect}")
print(f"")

# 2SLS
r_2sls = iv_2sls(Y_strong, D_strong, Z_strong, X_strong)
# LIML
r_liml = liml(Y_strong, D_strong, Z_strong, X_strong)
# Fuller(1)
r_fuller = fuller(Y_strong, D_strong, Z_strong, X_strong, alpha=1)

print(f"{'Estimator':<15} {'Estimate':>10} {'SE':>10} {'Bias':>10}")
print("-" * 50)
print(f"{'2SLS':<15} {r_2sls['estimate']:>10.4f} {r_2sls['se']:>10.4f} {r_2sls['estimate'] - true_effect:>10.4f}")
print(f"{'LIML':<15} {r_liml['estimate']:>10.4f} {r_liml['se']:>10.4f} {r_liml['estimate'] - true_effect:>10.4f}")
print(f"{'Fuller(1)':<15} {r_fuller['estimate']:>10.4f} {r_fuller['se']:>10.4f} {r_fuller['estimate'] - true_effect:>10.4f}")

In [ ]:
print("\n=== Comparison with WEAK Instrument ===")
print(f"True effect: {true_effect}")
print(f"")

# 2SLS
r_2sls_w = iv_2sls(Y_weak, D_weak, Z_weak, X_weak)
# LIML
r_liml_w = liml(Y_weak, D_weak, Z_weak, X_weak)
# Fuller(1)
r_fuller_w = fuller(Y_weak, D_weak, Z_weak, X_weak, alpha=1)

print(f"{'Estimator':<15} {'Estimate':>10} {'SE':>10} {'Bias':>10}")
print("-" * 50)
print(f"{'2SLS':<15} {r_2sls_w['estimate']:>10.4f} {r_2sls_w['se']:>10.4f} {r_2sls_w['estimate'] - true_effect:>10.4f}")
print(f"{'LIML':<15} {r_liml_w['estimate']:>10.4f} {r_liml_w['se']:>10.4f} {r_liml_w['estimate'] - true_effect:>10.4f}")
print(f"{'Fuller(1)':<15} {r_fuller_w['estimate']:>10.4f} {r_fuller_w['se']:>10.4f} {r_fuller_w['estimate'] - true_effect:>10.4f}")

### Key Observation

With a **weak instrument**:
- 2SLS is **biased toward OLS** (which itself is biased due to confounding)
- LIML and Fuller typically have **less bias**
- Standard errors may be understated for 2SLS

---

## Part 3: Anderson-Rubin Confidence Intervals

The **Anderson-Rubin (AR) test** provides valid inference regardless of instrument strength.

AR inverts a test of the reduced form: "Is there an effect at β₀?"
- Valid even with F < 10
- May produce wide or unbounded intervals with very weak instruments

In [ ]:
# Anderson-Rubin CI with strong instrument
print("=== Anderson-Rubin Confidence Intervals ===")
print(f"True effect: {true_effect}")
print(f"")

ar_strong = anderson_rubin_ci(Y_strong, D_strong, Z_strong, X_strong)
ar_weak = anderson_rubin_ci(Y_weak, D_weak, Z_weak, X_weak)

print("Strong Instrument:")
print(f"  2SLS CI:  [{r_2sls['ci_lower']:.4f}, {r_2sls['ci_upper']:.4f}]")
print(f"  AR CI:    [{ar_strong['ci_lower']:.4f}, {ar_strong['ci_upper']:.4f}]")
print(f"  Contains true effect: {ar_strong['ci_lower'] <= true_effect <= ar_strong['ci_upper']}")

print("\nWeak Instrument:")
print(f"  2SLS CI:  [{r_2sls_w['ci_lower']:.4f}, {r_2sls_w['ci_upper']:.4f}]")
print(f"  AR CI:    [{ar_weak['ci_lower']:.4f}, {ar_weak['ci_upper']:.4f}]")
print(f"  Contains true effect: {ar_weak['ci_lower'] <= true_effect <= ar_weak['ci_upper']}")

### AR CI Interpretation

- **Strong instrument**: AR CI similar to 2SLS CI (both valid)
- **Weak instrument**: AR CI typically wider but maintains valid coverage
- **Very weak instrument**: AR CI may be very wide or unbounded

---

## Part 4: Monte Carlo Demonstration

We run multiple simulations to show how estimators behave with weak instruments.

In [ ]:
# Monte Carlo simulation
n_sims = 200
true_effect = 0.1

results = {'2SLS': [], 'LIML': [], 'Fuller': []}

for i in range(n_sims):
    Y, D, Z, X = generate_iv_data(n=500, first_stage_strength=0.1, true_effect=true_effect, seed=i)
    
    try:
        results['2SLS'].append(iv_2sls(Y, D, Z, X)['estimate'])
        results['LIML'].append(liml(Y, D, Z, X)['estimate'])
        results['Fuller'].append(fuller(Y, D, Z, X, alpha=1)['estimate'])
    except:
        continue

print(f"Monte Carlo with WEAK instrument (F ≈ 5)")
print(f"True effect: {true_effect}")
print(f"Simulations: {n_sims}")
print(f"")
print(f"{'Estimator':<15} {'Mean':>10} {'SD':>10} {'Bias':>10} {'RMSE':>10}")
print("-" * 60)
for name, ests in results.items():
    ests = np.array(ests)
    bias = np.mean(ests) - true_effect
    rmse = np.sqrt(np.mean((ests - true_effect)**2))
    print(f"{name:<15} {np.mean(ests):>10.4f} {np.std(ests):>10.4f} {bias:>10.4f} {rmse:>10.4f}")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, ests) in zip(axes, results.items()):
    ax.hist(ests, bins=30, alpha=0.7, edgecolor='white')
    ax.axvline(x=true_effect, color='red', linestyle='--', label='True effect')
    ax.axvline(x=np.mean(ests), color='blue', linestyle='-', label='Mean estimate')
    ax.set_xlabel('Estimate')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name}\nBias = {np.mean(ests) - true_effect:.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Distribution of IV Estimates with Weak Instrument', y=1.02)
plt.show()

---

## Summary: Weak IV Decision Tree

```
Is F-statistic > 10?
│
├─ YES → Use 2SLS (standard inference valid)
│
└─ NO → Weak instrument concerns
    │
    ├─ For point estimation: Use LIML or Fuller(1)
    │   (less biased than 2SLS)
    │
    └─ For inference: Use Anderson-Rubin CI
        (valid coverage regardless of instrument strength)
```

### Key Takeaways

1. **Always report first-stage F-statistic** - F < 10 is a red flag
2. **2SLS is biased toward OLS** with weak instruments
3. **LIML and Fuller** have better finite-sample properties
4. **Anderson-Rubin CIs** provide valid inference regardless of instrument strength
5. **Wide AR CIs** honestly reflect uncertainty with weak identification

---

*Created: Session 98 - Causal Inference Mastery*